# BloomFace — MediaPipe Face Landmarker

Abre este notebook en VS Code local, selecciona un kernel de **Python 3.12** y pulsa **Ejecutar todo**. Prepara un entorno `.venv` en esta carpeta, instala las dependencias y recupera el modelo facial si falta. No necesita pesos de la CNN ni su preprocesamiento. La preparación comprueba el modelo real antes de abrir la ventana.

Esta integración conserva el generador floral original de la rama: su entorno DirectML y su activación se configuran por separado. Si está desactivado, puedes probar reconocimiento; no se generarán flores hasta habilitar y preparar ese entorno original.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

if os.name != 'nt':
    raise RuntimeError('Abre este notebook en Windows local para usar la webcam y la interfaz.')
candidatos = [Path.cwd(), Path.cwd() / 'BloomFace', Path.cwd().parent,
              Path(r'C:\Users\juand\OneDrive\Desktop\BloomFace')]
PROYECTO = next((p.resolve() for p in candidatos if (p / 'preparar_mediapipe.py').is_file()), None)
if PROYECTO is None:
    raise FileNotFoundError('Abre la carpeta completa de esta rama de BloomFace.')
SALIDA = PROYECTO / 'output'
SALIDA.mkdir(exist_ok=True)
PYTHON_LOCAL = PROYECTO / '.venv/Scripts/python.exe'
python_inicial = str(PYTHON_LOCAL) if PYTHON_LOCAL.is_file() else sys.executable
ENTORNO = dict(os.environ, PYTHONUTF8='1', PYTHONUNBUFFERED='1')
PREPARADO = False
print('Proyecto:', PROYECTO)


Proyecto: C:\Users\catao\OneDrive\Escritorio\PDI\Proyecto Final\BloomFace


In [3]:
ENTORNO["MPLBACKEND"] = "Agg"

## 1. Preparar MediaPipe y comprobar su carga

Espera a que termine. La primera ejecución puede descargar paquetes y el modelo facial. Los errores quedan en `output/preparacion_mediapipe.log`.


In [4]:
with subprocess.Popen([python_inicial, str(PROYECTO / 'preparar_mediapipe.py')],
                      cwd=PROYECTO, env=ENTORNO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding='utf-8', errors='replace') as proceso:
    for linea in proceso.stdout:
        print(linea, end='', flush=True)
    codigo = proceso.wait()
if codigo:
    raise RuntimeError('La preparación falló. Revisa la salida y output/preparacion_mediapipe.log.')
estado = json.loads((SALIDA / 'estado_mediapipe.json').read_text(encoding='utf-8'))
if not estado.get('preparado'):
    raise RuntimeError('El modelo no completó la comprobación.')
PREPARADO = True
print('Versiones:', estado['versiones'])


W0000 00:00:1795393862.109291   24548 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795393862.120794   16684 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795393862.141259   33004 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
{"mediapipe": "1.2.3", "PySide6": "6.12.0", "opencv-contrib-python": "5.1.0.95", "numpy": "2.4.6", "Pillow": "13.0.0"}
No broken requirements found.
MediaPipe preparado. Abre la cámara e inicia reconocimiento en la aplicación.
Versiones: {'mediapipe': '1.2.3', 'PySide6': '6.12.0', 'opencv-contrib-python': '5.1.0.95', 'numpy': '2.4.6', 'Pillow': '13.0.0'}


## 2. Abrir la aplicación

Cierra primero las ventanas anteriores de BloomFace. Pulsa **Abrir cámara → Iniciar reconocimiento**. Opcionalmente pulsa **Ajustar reposo (rostro relajado)** y mantén boca y cejas relajadas hasta completar 16 muestras con rostro detectado. Después prueba reposo, sonrisa, boca abierta y ceño. Las cifras son intensidades de movimiento, no probabilidades.

Para generación real, esta rama conserva las variables `BLOOMFACE_ENABLE_LOCAL_GENERATOR=1` y `BLOOMFACE_FLORAL_RUNTIME` del entorno original. No las activa automáticamente ni sustituye el generador por imágenes guardadas. Las instrucciones están en el README.


In [7]:
if not PREPARADO:
    raise RuntimeError('Completa primero la preparación de MediaPipe.')
anterior = globals().get('BLOOMFACE_PROCESO')
if anterior is not None and anterior.poll() is None:
    print('BloomFace ya está abierto; busca su ventana en la barra de tareas.')
else:
    registro_app = SALIDA / 'aplicacion_mediapipe.log'
    with registro_app.open('w', encoding='utf-8') as registro:
        BLOOMFACE_PROCESO = subprocess.Popen([str(PYTHON_LOCAL), '-m', 'bloomface'],
            cwd=PROYECTO, env=ENTORNO, stdin=subprocess.DEVNULL, stdout=registro,
            stderr=subprocess.STDOUT, creationflags=subprocess.CREATE_NEW_PROCESS_GROUP)
    time.sleep(2)
    if BLOOMFACE_PROCESO.poll() is not None:
        raise RuntimeError(registro_app.read_text(encoding='utf-8', errors='replace')[-5000:])
    print('BloomFace iniciado. Pulsa Abrir cámara → Iniciar reconocimiento.')


BloomFace iniciado. Pulsa Abrir cámara → Iniciar reconocimiento.


## Comprobación con webcam

Prueba los cuatro gestos, el habla, cambios de expresión, ausencia de rostro, pausa, reanudación y cierre. Después de reanudar pulsa de nuevo Iniciar reconocimiento. La aplicación exige tres decisiones consecutivas; las mezclas pueden quedar sin confirmar. Los umbrales iniciales aún requieren validación con participantes. Ajustar reposo no es entrenar una nueva red. El pequeño recorte gris es una referencia; el modelo recibe RGB.
